# Day 9

Mapping land cover of Paris across 4 boundaries
- Ville de Paris: the official city
- Métropole du Grand Paris: the political/administrative metropolis
- Unité urbaine: the physical city, the continuous built-up area
- Aire d’attraction: the functional city, everywhere people commute in from

### 1. QGIS

The following could have been automated in Python but I did it in QGIS instead to explore and visualise each layer

##### Layers 1 and 3: Base des unités urbaines 2020 (https://www.insee.fr/fr/information/4802589)
- Click “Fonds de cartes des unités urbaines 2020 au 1ᵉʳ janvier 2026” (zip, 8 Mo). Skip the “Base” file, which only has tables.
- Unzip and load both shapefiles into QGIS.
- uu2020_2026 (the outlines of each urban unit)
           Filter by "UU2020" = '00851' → layer 3: Unité urbaine
- com_uu2020_2026 (the communes)
           Filter by "CODGEO" = '75056' → layer 1: Ville de Paris
           Keep this layer: it’s also used for layer 2.

##### Layer 2: Intercommunalité, EPCI à fiscalité propre (https://www.insee.fr/fr/information/2510634)
- Click “Base des EPCI à fiscalité propre au 1ᵉʳ janvier 2026”. Skip “Base des EPT”, which leaves out Paris itself.
- Open it in Excel and go to the sheet listing each commune and its EPCI (Composition_communale).
- Delete the info rows above the column headers, so the header row is row 1.
- Save it as CSV UTF-8.
- Layer → Add Layer → Add Delimited Text Layer. Set No geometry (attribute only table) and untick
- Detect field types, so codes stay as text and keep their leading zeros.
- Right-click com_uu2020_2026 → Properties → Joins → green + button.
           Join layer: your CSV. Join field: CODGEO. Target field: CODGEO (the 5-digit commune code column).
           Tick Joined fields and select only EPCI and LIBEPCI. Tick Custom field name prefix and type mgp_.
           Click OK twice. Open the attribute table to check the new mgp_EPCI and mgp_LIBEPCI columns.
- Right-click the layer → Filter… → "mgp_LIBEPCI" = 'Métropole du Grand Paris'. The map should show only Paris and the inner suburbs (about 131 communes). Use straight quotes, not curly ones.
Vector → Geoprocessing Tools → Dissolve, input com_uu2020_2026, leave “Dissolve field(s)” empty → layer 2: Métropole du Grand Paris
- Clear the filter on com_uu2020_2026 afterwards.

##### Layer 4: Base des aires d’attraction des villes 2020 (https://www.insee.fr/fr/information/4803954)
- Click “Fonds de cartes des aires d’attraction des villes 2020 au 1ᵉʳ janvier 2026” (zip, 8 Mo).
- Unzip aav2020_2026 and load the shapefile.
- Filter by "AAV2020" = '001' → layer 4: Aire d’attraction

##### Export the four layers
- Right-click each one → Export → Save Features As… → GeoJSON, into the same folder as the script:

Paris_Ville de Paris_1.geojson
Paris_Métropole du Grand Paris_2.geojson
Paris_Unité urbaine_3.geojson
Paris_Aire d’attraction_4.geojson


##### 1. Import relevant libraries/packages

In [7]:
import math
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.merge import merge
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.features import sieve, geometry_mask
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

##### 2. Define the layers

In [41]:
LAYERS = {
    "Ville de Paris":           "Paris_Ville de Paris_1.geojson",
    "Métropole du Grand Paris": "Paris_Métropole du Grand Paris_2.geojson",
    "Unité urbaine":            "Paris_Unité urbaine_3.geojson",
    "Aire d'attraction":        "Paris_Aire d’attraction_4.geojson",
}

##### 3. Configurations

In [21]:
OUTER = "Aire d'attraction"  # the outermost layer

# WorldCover download
LOCAL_TIF = "worldcover_paris.tif"
BUFFER_KM = 5  # extra land cover around the outer boundary, for the map's soft edge
TILE_URL = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
            "ESA_WorldCover_10m_2021_v200_{tile}_Map.tif")

# Outputs
STATS_CSV = "paris_landcover_stats.csv"
LINES_PNG = "paris_landcover_lines.png"
MAP_PNG = "paris_landcover_map.png"
REUSE_STATS = True  # reuse the CSV if it exists (delete it to recount)

# WorldCover class codes grouped into poster-friendly categories
GROUPS = {
    "Built-up":        [50],
    "Cropland":        [40],
    "Tree cover":      [10],  # includes parks, street trees and gardens, not just forest
    "Grass & shrub":   [20, 30],
    "Water & wetland": [80, 90, 95],
    "Other":           [60, 70, 100],
}
MIN_PLOT_PCT = 1  # groups never reaching this % are left out of chart and legend

# Colours. Chart lines are stronger (checked for colourblind separation);
# map fills are softer versions of the same hues, with only built-up kept dark.
CHART_COLOURS = {
    "Built-up": "#7a4e36", "Cropland": "#c9970c", "Tree cover": "#21857a",
    "Grass & shrub": "#7fae2e", "Water & wetland": "#3f8fc4", "Other": "#a8a29a",
}
MAP_COLOURS = {
    "Built-up": "#4a3226", "Cropland": "#ead18a", "Tree cover": "#6f9e8c",
    "Grass & shrub": "#c3d39a", "Water & wetland": "#9cc6de", "Other": "#ddd6c8",
}
BG = "#f5f1e8"      # cream background, used by both outputs
INK = "#2b2b2b"     # text and boundary lines
INK_SOFT = "#666666"

# Boundary line styles, innermost to outermost
BOUNDARY_STYLE = {
    "Ville de Paris":           dict(lw=1.8, ls="-"),
    "Métropole du Grand Paris": dict(lw=1.3, ls="-"),
    "Unité urbaine":            dict(lw=1.3, ls=(0, (5, 2.5))),
    "Aire d'attraction":        dict(lw=1.6, ls=(0, (1, 2))),
}

# Map legend text for each boundary (set to {} to show just the names)
BOUNDARY_LABELS = {
    "Ville de Paris":           "Ville de Paris: the official city",
    "Métropole du Grand Paris": "Métropole du Grand Paris: the administrative metropolis",
    "Unité urbaine":            "Unité urbaine: the continuous built-up area",
    "Aire d'attraction":        "Aire d'attraction: where people commute in from",
}

# Map design
MAP_RES_M = 40      # map pixel size in metres (smaller = sharper but slower)
SIEVE_PX = 8        # merge specks smaller than this many map pixels (0 = off)
FADE = 0.85         # how much land cover outside the outer boundary fades out
MAP_PAD_KM = 3      # space around the outer boundary (keeps the raster edge out of view)
INSET_BOX = [0.0, 0.0, 0.30, 0.30]  # zoom inset position in map axes (x, y, w, h)
SCALE_KM = 20
TITLE = "Where does Paris end?"
SUBTITLE = "Land cover inside four official definitions of the city"
CHART_HEADING = "The wider the boundary, the more rural Paris becomes"
CREDIT = ("Land cover: © ESA WorldCover project 2021\n"
          "Boundaries: INSEE (communes, EPCI, unités urbaines 2020, "
          "aires d'attraction des villes 2020)")

# Poster: map on top, line chart + table underneath
POSTER_PNG = "paris_landcover_poster.png"
POSTER_SIZE = (14, 20)  # inches (at 300 dpi = 4200 x 6000 px)

BLOCK = 1024  # rows per block when counting (keeps memory low)

##### 4. Helper functions

In [25]:
# ================================================================ data
def load_layers():
    missing = [p for p in LAYERS.values() if not os.path.exists(p)]
    if missing:
        here = "\n  ".join(sorted(f for f in os.listdir(".") if f.endswith(".geojson")))
        raise FileNotFoundError(f"Can't find: {missing}\nGeoJSONs in this folder:\n  {here}")
    return {n: gpd.read_file(p).to_crs(2154).dissolve() for n, p in LAYERS.items()}


def tile_names(bounds):
    """WorldCover tile names (3x3 degree, named by lower-left corner) covering a bbox."""
    minx, miny, maxx, maxy = bounds
    names = []
    for lat in range(math.floor(miny / 3) * 3, math.ceil(maxy / 3) * 3, 3):
        for lon in range(math.floor(minx / 3) * 3, math.ceil(maxx / 3) * 3, 3):
            ns = f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}"
            ew = f"{'E' if lon >= 0 else 'W'}{abs(lon):03d}"
            names.append(ns + ew)
    return names


def ensure_worldcover(outer_gdf):
    """Download the needed part of WorldCover to LOCAL_TIF, unless it already exists."""
    if os.path.exists(LOCAL_TIF):
        print(f"Using existing {LOCAL_TIF}")
        return
    area = outer_gdf.buffer(BUFFER_KM * 1000).to_crs(4326)
    bounds = tuple(area.total_bounds)
    tiles = tile_names(bounds)
    print(f"Downloading WorldCover {', '.join(tiles)} ({OUTER} + {BUFFER_KM} km) ...")
    sources = [rasterio.open(TILE_URL.format(tile=t)) for t in tiles]
    try:
        data, transform = merge(sources, bounds=bounds, nodata=0)
        profile = sources[0].profile.copy()
        try:
            colormap = sources[0].colormap(1)
        except ValueError:
            colormap = None
    finally:
        for s in sources:
            s.close()
    profile.update(driver="GTiff", height=data.shape[1], width=data.shape[2],
                   transform=transform, nodata=0, compress="deflate",
                   tiled=True, blockxsize=512, blockysize=512, BIGTIFF="IF_SAFER")
    with rasterio.open(LOCAL_TIF, "w", **profile) as dst:
        dst.write(data)
        if colormap:
            dst.write_colormap(1, colormap)
    print(f"Saved {LOCAL_TIF}")


def class_areas_km2(geom_4326):
    """Area (km²) of each WorldCover class inside a geometry, weighting pixels by latitude."""
    totals = np.zeros(256)
    with rasterio.open(LOCAL_TIF) as src:
        data, transform = mask(src, [geom_4326], crop=True, nodata=0)
    data = data[0]
    res_x, res_y, top = transform.a, -transform.e, transform.f
    for r0 in range(0, data.shape[0], BLOCK):
        block = data[r0:r0 + BLOCK]
        lat = top - (r0 + block.shape[0] / 2) * res_y
        px_km2 = (res_x * 111.320 * np.cos(np.radians(lat))) * (res_y * 110.574)
        totals += np.bincount(block.ravel(), minlength=256) * px_km2
    totals[0] = 0
    return totals


def compute_stats(gdfs):
    if REUSE_STATS and os.path.exists(STATS_CSV):
        old = pd.read_csv(STATS_CSV)
        # CSVs from the earlier script called tree cover "Forest"
        old = old.rename(columns={"Forest": "Tree cover", "Forest %": "Tree cover %"})
        needed = ["name", "Total km²"] + list(GROUPS) + [f"{g} %" for g in GROUPS]
        missing = [c for c in needed if c not in old.columns]
        if not missing:
            print(f"Using existing {STATS_CSV}")
            old.round(2).to_csv(STATS_CSV, index=False)  # save with the new column names
            return old
        print(f"{STATS_CSV} is missing {missing}, so recounting ...")
    rows = []
    for name, gdf in gdfs.items():
        print(f"Counting: {name} ...")
        totals = class_areas_km2(gdf.to_crs(4326).geometry.union_all())
        row = {"name": name}
        row.update({g: totals[codes].sum() for g, codes in GROUPS.items()})
        row["Total km²"] = sum(row[g] for g in GROUPS)
        for g in GROUPS:
            row[f"{g} %"] = 100 * row[g] / row["Total km²"] if row["Total km²"] else 0
        rows.append(row)
    df = pd.DataFrame(rows)
    df.round(2).to_csv(STATS_CSV, index=False)
    print(f"Saved {STATS_CSV}")
    return df



# ================================================================ line chart + table
X_LABELS = ["Ville de\nParis", "Métropole du\nGrand Paris", "Unité\nurbaine",
            "Aire\nd'attraction"]


def draw_lines(ax, tab, df, shown, fs=1.0):
    """Line chart on `ax`, table of exact numbers on `tab` (the table doubles as legend)."""
    plot_df = df.set_index("name").loc[list(LAYERS)]
    x = np.arange(len(plot_df))

    ax.set_facecolor(BG)
    for g in shown:
        ax.plot(x, plot_df[f"{g} %"].values, color=CHART_COLOURS[g], linewidth=2.5 * fs,
                marker="o", markersize=7 * fs, markeredgecolor=BG,
                markeredgewidth=1.5 * fs, zorder=3)
    ax.set_xticks(x, X_LABELS, fontsize=10 * fs)
    ax.set_xlim(-0.25, x[-1] + 0.25)
    top = max(plot_df[f"{g} %"].max() for g in shown)
    ax.set_ylim(0, min(100, np.ceil((top + 5) / 10) * 10))
    ax.set_ylabel("% of area", color=INK_SOFT, fontsize=10 * fs)
    ax.grid(axis="y", color="#e2ddd2", linewidth=0.8, zorder=0)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.spines["bottom"].set_color("#999999")
    ax.tick_params(axis="y", length=0, colors=INK_SOFT, labelsize=10 * fs)
    ax.tick_params(axis="x", colors=INK)

    tab.axis("off")
    tab.set_xlim(0, 1)
    tab.set_ylim(0, 1)
    col_x = [0.42, 0.575, 0.73, 0.885]
    row_h = 0.80 / (len(shown) + 2)
    y = 0.94
    for cx, lab in zip(col_x, X_LABELS):
        tab.text(cx, y, lab, ha="center", va="center", fontsize=9.5 * fs, color=INK_SOFT)
    y -= row_h * 0.9
    tab.plot([0, 0.96], [y + row_h * 0.45] * 2, color="#c8c2b5", lw=0.8)
    for g in shown:
        tab.plot([0.0, 0.05], [y, y], color=CHART_COLOURS[g], lw=2.5 * fs,
                 solid_capstyle="round")
        tab.plot(0.025, y, "o", color=CHART_COLOURS[g], ms=6 * fs, mec=BG, mew=1.2)
        tab.text(0.07, y, g, va="center", fontsize=10.5 * fs, color=INK)
        for cx, v in zip(col_x, plot_df[f"{g} %"].values):
            tab.text(cx, y, f"{v:.1f}%", ha="center", va="center", fontsize=10.5 * fs,
                     color=INK)
        y -= row_h
    tab.plot([0, 0.96], [y + row_h * 0.5] * 2, color="#c8c2b5", lw=0.8)
    tab.text(0.07, y, "Total area (km²)", va="center", fontsize=10.5 * fs, color=INK_SOFT)
    for cx, v in zip(col_x, plot_df["Total km²"].values):
        tab.text(cx, y, f"{v:,.0f}", ha="center", va="center", fontsize=10.5 * fs,
                 color=INK_SOFT)


def make_lines_png(df, shown):
    fig, (ax, tab) = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=BG,
                                  gridspec_kw={"width_ratios": [1.25, 1], "wspace": 0.08})
    draw_lines(ax, tab, df, shown)
    plt.subplots_adjust(left=0.06, right=0.99, top=0.95, bottom=0.14)
    plt.savefig(LINES_PNG, dpi=200, facecolor=BG)
    plt.close(fig)
    print(f"Saved {LINES_PNG}")


# ================================================================ map
def prepare_map_raster(gdfs):
    """Reproject WorldCover to Lambert-93, colour it, and fade everything outside OUTER."""
    print("Preparing map raster ...")
    with rasterio.open(LOCAL_TIF) as src:
        transform, width, height = calculate_default_transform(
            src.crs, "EPSG:2154", src.width, src.height, *src.bounds, resolution=MAP_RES_M)
        lc = np.zeros((height, width), dtype=np.uint8)
        reproject(rasterio.band(src, 1), lc, dst_transform=transform, dst_crs="EPSG:2154",
                  resampling=Resampling.mode, src_nodata=0, dst_nodata=0)
    if SIEVE_PX:
        lc = sieve(lc, size=SIEVE_PX)

    bg = np.array(to_rgb(BG), dtype=np.float32)
    lut = np.tile(bg, (256, 1))
    for g, codes in GROUPS.items():
        lut[codes] = to_rgb(MAP_COLOURS[g])
    rgb = lut[lc]
    outside = geometry_mask([gdfs[OUTER].geometry.union_all()], out_shape=lc.shape,
                            transform=transform)
    rgb[outside] = rgb[outside] * (1 - FADE) + bg * FADE

    left, top = transform.c, transform.f
    extent = [left, left + width * transform.a, top + height * transform.e, top]
    return rgb, extent


def draw_boundaries(ax, gdfs, scale=1.0):
    for name, gdf in gdfs.items():
        s = BOUNDARY_STYLE[name]
        # a pale halo underneath keeps the lines readable over dark built-up areas
        gdf.boundary.plot(ax=ax, color=BG, linewidth=(s["lw"] + 1.6) * scale,
                          alpha=0.7, zorder=4)
        gdf.boundary.plot(ax=ax, color=INK, linewidth=s["lw"] * scale,
                          linestyle=s["ls"], zorder=5)


def draw_map(ax, gdfs, rgb, extent, fs=1.0):
    """The land-cover map, boundaries, zoom inset and scale bar, on `ax`."""
    ax.imshow(rgb, extent=extent, interpolation="nearest")
    draw_boundaries(ax, gdfs, scale=fs)
    minx, miny, maxx, maxy = gdfs[OUTER].total_bounds
    pad = MAP_PAD_KM * 1000
    left, right, bottom, top = minx - pad, maxx + pad, miny - pad, maxy + pad
    ax.set_xlim(left, right)
    ax.set_ylim(bottom, top)
    ax.set_aspect("equal")
    ax.set_axis_off()

    # Zoom inset on inner Paris, where the two smallest boundaries are too small to read
    ins = ax.inset_axes(INSET_BOX)
    ins.imshow(rgb, extent=extent, interpolation="nearest")
    draw_boundaries(ins, gdfs, scale=1.2 * fs)
    iminx, iminy, imaxx, imaxy = gdfs["Métropole du Grand Paris"].total_bounds
    ipad = 3000
    ins.set_xlim(iminx - ipad, imaxx + ipad)
    ins.set_ylim(iminy - ipad, imaxy + ipad)
    ins.set_aspect("equal")
    ins.set_xticks([])
    ins.set_yticks([])
    ins.set_xlabel("")
    ins.set_ylabel("")
    for sp in ins.spines.values():
        sp.set_edgecolor(INK)
        sp.set_linewidth(0.8)
    ax.indicate_inset_zoom(ins, edgecolor=INK, alpha=0.6)
    ins.text(0.04, 0.96, "Inner Paris", transform=ins.transAxes, va="top",
             fontsize=9 * fs, color=INK, bbox=dict(fc=BG, ec="none", alpha=0.85, pad=2))

    # Scale bar, bottom right
    L = SCALE_KM * 1000
    x0 = right - (right - left) * 0.04 - L
    y0 = bottom + (top - bottom) * 0.04
    tick = (top - bottom) * 0.006
    ax.plot([x0, x0 + L], [y0, y0], color=INK, lw=1.5, solid_capstyle="butt", zorder=6)
    for xx in (x0, x0 + L / 2, x0 + L):
        ax.plot([xx, xx], [y0, y0 + tick], color=INK, lw=1, zorder=6)
    ax.text(x0, y0 + tick * 2.5, "0", ha="center", fontsize=9 * fs, color=INK)
    ax.text(x0 + L, y0 + tick * 2.5, f"{SCALE_KM} km", ha="center", fontsize=9 * fs,
            color=INK)
    ax.set_xlabel("")
    ax.set_ylabel("")


def add_map_legends(fig, shown, y_landcover, y_lines, fs=1.0):
    lc_handles = [Patch(facecolor=MAP_COLOURS[g], edgecolor="none", label=g) for g in shown]
    lc_legend = fig.legend(handles=lc_handles, loc="center", bbox_to_anchor=(0.5, y_landcover),
                           ncol=len(lc_handles), frameon=False, fontsize=10.5 * fs,
                           handlelength=1.4, handleheight=1.0, columnspacing=1.6)
    fig.add_artist(lc_legend)
    line_handles = [Line2D([0], [0], color=INK, lw=BOUNDARY_STYLE[n]["lw"] * fs,
                           linestyle=BOUNDARY_STYLE[n]["ls"],
                           label=BOUNDARY_LABELS.get(n, n)) for n in LAYERS]
    two_rows = bool(BOUNDARY_LABELS)  # long labels go on two rows
    fig.legend(handles=line_handles, loc="center", bbox_to_anchor=(0.5, y_lines),
               ncol=2 if two_rows else len(line_handles), frameon=False,
               fontsize=10.5 * fs, handlelength=3, columnspacing=2.5)


def make_map_png(gdfs, shown, rgb, extent):
    fig = plt.figure(figsize=(12, 13), facecolor=BG)
    ax = fig.add_axes([0.03, 0.15, 0.94, 0.74])
    draw_map(ax, gdfs, rgb, extent)
    fig.text(0.03, 0.955, TITLE, fontsize=26, fontweight="bold", color=INK, va="center")
    fig.text(0.03, 0.918, SUBTITLE, fontsize=13, color=INK_SOFT, va="center")
    add_map_legends(fig, shown, y_landcover=0.115, y_lines=0.068)
    fig.text(0.5, 0.012, CREDIT, ha="center", va="bottom", fontsize=8, color=INK_SOFT,
             linespacing=1.5)
    plt.savefig(MAP_PNG, dpi=300, facecolor=BG)
    plt.close(fig)
    print(f"Saved {MAP_PNG}")


# ================================================================ poster
def make_poster(gdfs, df, shown, rgb, extent):
    """Map on top, legends under it, then the line chart and table, then the credit."""
    fs = 1.15  # text a little larger on the poster
    fig = plt.figure(figsize=POSTER_SIZE, facecolor=BG)
    m = 0.045  # side margin

    fig.text(m, 0.972, TITLE, fontsize=34, fontweight="bold", color=INK, va="center")
    fig.text(m, 0.948, SUBTITLE, fontsize=16, color=INK_SOFT, va="center")

    map_ax = fig.add_axes([m, 0.40, 1 - 2 * m, 0.53])
    draw_map(map_ax, gdfs, rgb, extent, fs=fs)
    add_map_legends(fig, shown, y_landcover=0.383, y_lines=0.352, fs=fs)

    # Divider and chart heading
    fig.add_artist(Line2D([m, 1 - m], [0.318, 0.318], color="#c8c2b5", lw=1,
                          transform=fig.transFigure))
    fig.text(m, 0.298, CHART_HEADING, fontsize=18, fontweight="bold", color=INK,
             va="center")
    pdf = df.set_index("name").loc[list(LAYERS)]
    if "Built-up" in shown and "Cropland" in shown:
        sub = (f"Built-up land falls from {pdf['Built-up %'].iloc[0]:.0f}% of the official "
               f"city to {pdf['Built-up %'].iloc[-1]:.0f}% of the commuter region, while "
               f"cropland rises from {pdf['Cropland %'].iloc[0]:.0f}% to "
               f"{pdf['Cropland %'].iloc[-1]:.0f}%.")
        fig.text(m, 0.278, sub, fontsize=12.5, color=INK_SOFT, va="center")

    chart_ax = fig.add_axes([m + 0.04, 0.075, 0.45, 0.18])
    tab_ax = fig.add_axes([0.555, 0.075, 1 - m - 0.555, 0.19])
    draw_lines(chart_ax, tab_ax, df, shown, fs=fs)

    fig.text(0.5, 0.012, CREDIT, ha="center", va="bottom", fontsize=9, color=INK_SOFT,
             linespacing=1.5)
    plt.savefig(POSTER_PNG, dpi=300, facecolor=BG)
    plt.close(fig)
    print(f"Saved {POSTER_PNG}")

In [27]:
# ================================================================ main
def main():
    gdfs = load_layers()
    ensure_worldcover(gdfs[OUTER])
    df = compute_stats(gdfs)
    shown = [g for g in GROUPS if df[f"{g} %"].max() >= MIN_PLOT_PCT]
    print(df[["name", "Total km²"] + [f"{g} %" for g in GROUPS]].round(1).to_string())
    make_lines_png(df, shown)
    rgb, extent = prepare_map_raster(gdfs)
    make_map_png(gdfs, shown, rgb, extent)
    make_poster(gdfs, df, shown, rgb, extent)

##### 5. Run the function

In [61]:
if __name__ == "__main__":
    main()

Using existing worldcover_paris.tif
Using existing paris_landcover_stats.csv
                       name  Total km²  Built-up %  Cropland %  Tree cover %  Grass & shrub %  Water & wetland %  Other %
0            Ville de Paris      104.1        73.2         0.1          22.2              2.6                1.9      0.0
1  Métropole du Grand Paris      809.0        60.4         2.0          30.8              5.0                1.7      0.2
2             Unité urbaine     2819.3        35.8        12.7          41.0              8.5                1.7      0.2
3         Aire d'attraction    18896.0         8.4        49.6          33.0              8.0                0.8      0.1
Saved paris_landcover_lines.png
Preparing map raster ...
Saved paris_landcover_map.png
Saved paris_landcover_poster.png
